# Step 4 — Fused FlashAttention-1 (v4_fused) — run-of-record (vast.ai T4)

Single fused pass: warp-per-row, staged K/V in smem, register-resident O, online softmax with
the **O-rescale**. Keeps v3's S-off-HBM property AND restores v2's GEMM-like parallelism.

**Thesis to test:** v4 should *beat v2* in wall-clock (reversing v3's 3-7x regression) while
keeping the +17 MB peak-memory footprint (S still gone). Counter-free throughout (ncu blocked
on cloud rentals): peak-memory for the S proof, torch-profiler CUPTI for per-kernel timing.


## 0. Bootstrap the vast.ai CUDA devel image (idempotent)
The bare CUDA devel image has **no `python` symlink, no torch, and we start outside the repo** —
the three things that failed on the first run. This cell fixes all of them; safe to re-run.


In [1]:
# (a) Route EVERYTHING through the Jupyter kernel's own interpreter. `!python` (a shell
#     subprocess) and an in-kernel `import torch` are otherwise DIFFERENT pythons on this
#     image, so torch installed for one is invisible to the other. Point the `python`
#     symlink at sys.executable and install into sys.executable -> one python, both paths.
import subprocess, sys, os
KPY = sys.executable  # the kernel running this notebook
subprocess.run(['ln','-sf', KPY, '/usr/local/bin/python'])
print('python ->', KPY)

# (b) torch (cu124) into the KERNEL interpreter if missing, plus build deps.
try:
    import torch  # noqa: F401
    print('torch already present:', torch.__version__)
except ModuleNotFoundError:
    subprocess.run([KPY,'-m','pip','install','-q','torch',
                    '--index-url','https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([KPY,'-m','pip','install','-q','ninja','pytest'], check=True)

# (c) locate an existing repo checkout (walk up from cwd) or clone, then chdir in.
#     Works whether you uploaded the bare .ipynb or run it from inside a clone.
REPO='https://github.com/gkienpham-cmd/flashattention-cuda.git'
def _find_repo(p):
    while True:
        if os.path.isdir(os.path.join(p,'.git')) and os.path.isdir(os.path.join(p,'fa_kernels')):
            return p
        nxt = os.path.dirname(p)
        if nxt == p: return None
        p = nxt
root = _find_repo(os.getcwd())
if root is None:
    if not os.path.isdir('flashattention-cuda/.git'):
        subprocess.run(['git','clone',REPO], check=True)
    root = os.path.abspath('flashattention-cuda')
os.chdir(root)
subprocess.run(['git','pull','origin','main'], check=True)
print('cwd =', os.getcwd())


python -> /venv/main/bin/python


Updating df818ae..69025bb
Fast-forward
 notebooks/step4_run_of_record.ipynb | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
cwd = /flashattention-cuda


From https://github.com/gkienpham-cmd/flashattention-cuda
 * branch            main       -> FETCH_HEAD
   df818ae..69025bb  main       -> origin/main


## 1. Confirm the toolchain + GPU


In [2]:
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Tue_Oct_29_23:50:19_PDT_2024
Cuda compilation tools, release 12.6, V12.6.85
Build cuda_12.6.r12.6/compiler.35059454_0


In [3]:
!python -c "import torch; print('cuda_ok', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0), '|', torch.version.cuda)"


/venv/main/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
cuda_ok True | Tesla T4 | 12.4


## 2. Predict the roofline BEFORE running (record the floor — the deliverable is the distance)


In [4]:
!python -m roofline.predict --arch sm_75 --shape 1x8x8192x64 --precision fp32
!python -m roofline.predict --arch sm_75 --shape 1x8x2048x128 --precision fp32


arch        : Tesla T4 (sm_75)
shape       : B=1 H=8 N=8192 d=64  precision=fp32  materialize_S=False  tile=1x1
LIMITER     : MMA   (predicted lower bound 16.968 ms)
  t_mma     :   16.968 ms   util 100.0%
  t_hbm     :    0.210 ms   util   1.2%
  t_mufu    :    0.530 ms   util   3.1%
intensity   : 2048.0 FLOP/byte   (arch ridge 25.3; ABOVE -> compute-bound)
arch        : Tesla T4 (sm_75)
shape       : B=1 H=8 N=2048 d=128  precision=fp32  materialize_S=False  tile=1x1
LIMITER     : MMA   (predicted lower bound 2.121 ms)
  t_mma     :    2.121 ms   util 100.0%
  t_hbm     :    0.105 ms   util   4.9%
  t_mufu    :    0.033 ms   util   1.6%
intensity   : 512.0 FLOP/byte   (arch ridge 25.3; ABOVE -> compute-bound)


## 3. Build smoke (JIT compile v4 + one forward). A clean compile here = the kernel built.


In [5]:
# Clear any stale JIT cache from a prior build, then compile v4 via one forward.
import shutil, os, glob
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v4_fused')):
    shutil.rmtree(d, ignore_errors=True)
import torch
from fa_kernels import attention
q=torch.randn(1,8,512,64,device='cuda'); k=torch.randn_like(q); v=torch.randn_like(q)
out=attention(q,k,v,backend='v4_fused'); torch.cuda.synchronize()
print('v4 built + ran, out shape', tuple(out.shape))


/venv/main/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
Using /root/.cache/torch_extensions/py312_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py312_cu124/fa_v4_fused...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py312_cu124/fa_v4_fused/build.ninja...
/venv/main/lib/python3.12/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fa_v4_fused...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JO

[1/3] c++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fa_v4_fused -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /venv/main/lib/python3.12/site-packages/torch/include -isystem /venv/main/lib/python3.12/site-packages/torch/include/torch/csrc/api/include -isystem /venv/main/lib/python3.12/site-packages/torch/include/TH -isystem /venv/main/lib/python3.12/site-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /venv/main/include/python3.12 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /flashattention-cuda/kernels/v4_fused/binding.cpp -o binding.o 
[2/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output fused_attention.cuda.o.d -DTORCH_EXTENSION_NAME=fa_v4_fused -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /venv/main/lib/python3.12/site-packages/

Loading extension module fa_v4_fused...


## 4. Correctness vs SDPA (atol/rtol 1e-4) — full sweep + the long-N O-rescale stability test


In [6]:
!python -m pytest tests/test_correctness.py -k v4_fused -v


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /flashattention-cuda
configfile: pyproject.toml
plugins: anyio-4.13.0
collected 58 items / 41 deselected / 17 selected                               

tests/test_correctness.py::test_matches_sdpa[False-1-4-128-64-v4_fused] PASSED [  5%]
tests/test_correctness.py::test_matches_sdpa[False-2-8-512-64-v4_fused] PASSED [ 11%]
tests/test_correctness.py::test_matches_sdpa[False-1-8-512-128-v4_fused] PASSED [ 17%]
tests/test_correctness.py::test_matches_sdpa[False-1-2-2048-64-v4_fused] PASSED [ 23%]
tests/test_correctness.py::test_matches_sdpa[False-1-2-130-64-v4_fused] PASSED [ 29%]
tests/test_correctness.py::test_matches_sdpa[False-1-2-100-128-v4_fused] PASSED [ 35%]
tests/test_correctness.py::test_matches_sdpa[True-1-4-128-64-v4_fused] PASSED [ 41%]
tests/test_correctness.py::test_matches_sdpa[

## 5. S-elimination proof (peak memory) — v4 must match v3's +17 MB, NOT v2's +2164 MB
The O-rescale lets v4 stay single-pass with no S and no per-row (m,l) HBM scratch — even less
than v3's two tiny stats arrays.


In [7]:
%%writefile mem_check.py
import torch
from fa_kernels import attention
B,H,N,d = 1,8,8192,64
for backend in ["v2_tiled", "v3_online", "v4_fused"]:
    q=torch.randn(B,H,N,d,device='cuda'); k=torch.randn(B,H,N,d,device='cuda'); v=torch.randn(B,H,N,d,device='cuda')
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); base=torch.cuda.memory_allocated()
    out=attention(q,k,v,backend=backend); torch.cuda.synchronize()
    print(f"{backend}: peak +{(torch.cuda.max_memory_allocated()-base)/1e6:.1f} MB  (a materialized S = {B*H*N*N*4/1e6:.0f} MB)")
    del q,k,v,out; torch.cuda.empty_cache()


Writing mem_check.py


In [8]:
!python mem_check.py


/venv/main/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
Using /root/.cache/torch_extensions/py312_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py312_cu124/fa_v2_tiled...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py312_cu124/fa_v2_tiled/build.ninja...
/venv/main/lib/python3.12/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fa_v2_tiled...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JO

## 6. CUPTI per-kernel trace — v4 is a SINGLE fused kernel; measure its distance from the 17 ms floor
v3 was pass2-dominated (88.6%) and 151x above the floor. v4 collapses to one kernel — read its
total CUDA time and compare to the 16.97 ms MMA lower bound.


In [9]:
%%writefile prof_check.py
import torch
from torch.profiler import profile, ProfilerActivity
from fa_kernels import attention
B,H,N,d = 1,8,8192,64
q=torch.randn(B,H,N,d,device='cuda'); k=torch.randn(B,H,N,d,device='cuda'); v=torch.randn(B,H,N,d,device='cuda')
for _ in range(3): attention(q,k,v,backend="v4_fused")   # warmup + JIT
torch.cuda.synchronize()
with profile(activities=[ProfilerActivity.CUDA]) as prof:
    for _ in range(10): attention(q,k,v,backend="v4_fused")
    torch.cuda.synchronize()
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))


Writing prof_check.py


In [10]:
!python prof_check.py


/venv/main/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
Using /root/.cache/torch_extensions/py312_cu124 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py312_cu124/fa_v4_fused/build.ninja...
/venv/main/lib/python3.12/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fa_v4_fused...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
ninja: no work to do.
Loading extension module fa_v4_fused...
------------------

## 7. Bench vs SDPA — the headline. Run v4, then v3 and v2 for the apples-to-apples comparison.
Win condition: v4/SDPA speedup >= v2/SDPA at matching shapes (v4 beats v2), and v4 >> v3.


In [11]:
!python -m bench.harness --backend v4_fused --precision fp32


/venv/main/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
# device: Tesla T4 (sm_75)  clock~585/1590MHz  backend=v4_fused  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
Using /root/.cache/torch_extensions/py312_cu124 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py312_cu124/fa_v4_fused/build.ninja...
/venv/main/lib/python3.12/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fa_v4_fused...
Al

In [12]:
!python -m bench.harness --backend v3_online --precision fp32


/venv/main/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
# device: Tesla T4 (sm_75)  clock~585/1590MHz  backend=v3_online  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
Using /root/.cache/torch_extensions/py312_cu124 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py312_cu124/fa_v3_online/build.ninja...
/venv/main/lib/python3.12/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fa_v3_online...

In [13]:
!python -m bench.harness --backend v2_tiled --precision fp32


/venv/main/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
# device: Tesla T4 (sm_75)  clock~585/1590MHz  backend=v2_tiled  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
Using /root/.cache/torch_extensions/py312_cu124 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py312_cu124/fa_v2_tiled/build.ninja...
/venv/main/lib/python3.12/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fa_v2_tiled...
Al

## 8. (Optional) causal sweep — exercises the early-out mask path in the fused loop


In [14]:
!python -m bench.harness --backend v4_fused --precision fp32 --causal


/venv/main/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
# device: Tesla T4 (sm_75)  clock~585/1590MHz  backend=v4_fused  precision=fp32  causal=True
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
Using /root/.cache/torch_extensions/py312_cu124 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py312_cu124/fa_v4_fused/build.ninja...
/venv/main/lib/python3.12/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fa_v4_fused...
All